# 02 — Prepare reproducible run logging

Run this before the live Gemini notebook. It creates a fixed configuration and helpers that validate one record per Dexter task. It does not call Gemini.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
ANSWERS_DIR = RUNS_DIR / 'answers'
RUNS_DIR.mkdir(exist_ok=True)
ANSWERS_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
if not MANIFEST_PATH.is_file():
    raise RuntimeError('Run 01_build_task_manifest.ipynb first.')
tasks = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line]
print(f'Loaded {len(tasks)} fixed pilot tasks.')

In [ ]:
# Fill the exact model identifier after selecting it in Dexter.
# Do not change this file during comparable conditions.
RUN_CONFIG = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': str(Path.home() / 'Downloads' / 'dataset1.xlsx'),
    'task_count': len(tasks),
    'model_provider': 'google',
    'model_name': 'FILL_IN_EXACT_GEMINI_MODEL_NAME',
    'temperature': 0,
    'max_iterations': 'FILL_IN_DEXTER_SETTING',
    'web_access': False,
    'prompt_template': 'Question: {question}\n\nData supplied for this task only:\n{provided_data}',
    'api_key_present': bool(os.environ.get('GOOGLE_API_KEY')),
}
CONFIG_PATH = RUNS_DIR / 'run_config.json'
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print(f'Wrote {CONFIG_PATH}')
print('Gemini key present in this kernel:', RUN_CONFIG['api_key_present'])
print('The key itself is never written to disk.')

In [ ]:
REQUIRED_RESULT_FIELDS = {
    'run_id', 'condition', 'selection_strategy', 'top_k', 'task_id', 'repeat',
    'completed', 'tool_calls', 'input_tokens', 'output_tokens', 'latency_ms',
    'retrieval_overhead_ms', 'retrieved_memory_ids', 'retrieval_relevant',
    'repeated_error', 'transfer_success', 'stale_memory_error', 'answer_path',
}
VALID_CONDITIONS = {'none', 'raw_trace', 'summary', 'structured_lesson'}
VALID_STRATEGIES = {'relevance', 'recency', 'none'}
RESULTS_PATH = RUNS_DIR / 'results.jsonl'

def validate_result(record: dict) -> None:
    missing = REQUIRED_RESULT_FIELDS - record.keys()
    if missing:
        raise ValueError(f'Missing fields: {sorted(missing)}')
    if record['condition'] not in VALID_CONDITIONS:
        raise ValueError('Unknown memory condition')
    if record['selection_strategy'] not in VALID_STRATEGIES:
        raise ValueError('Unknown retrieval strategy')
    if record['task_id'] not in {task['task_id'] for task in tasks}:
        raise ValueError('Task is not in the fixed manifest')
    if record['condition'] == 'none' and record['retrieved_memory_ids']:
        raise ValueError('No-memory baseline cannot retrieve memories')
    if record['tool_calls'] is not None and record['tool_calls'] < 0:
        raise ValueError('tool_calls cannot be negative')

def append_result(record: dict, answer_text: str) -> None:
    validate_result(record)
    answer_path = PROJECT_ROOT / record['answer_path']
    answer_path.parent.mkdir(parents=True, exist_ok=True)
    answer_path.write_text(answer_text, encoding='utf-8')
    existing_ids = set()
    if RESULTS_PATH.exists():
        existing_ids = {json.loads(line)['run_id'] for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line}
    if record['run_id'] in existing_ids:
        raise ValueError(f'Duplicate run_id: {record["run_id"]}')
    with RESULTS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

In [ ]:
# Schema-only dry run. It validates a record but does not save a result.
example = {
    'run_id': 'DO_NOT_SAVE-example', 'condition': 'none', 'selection_strategy': 'none',
    'top_k': 0, 'task_id': tasks[0]['task_id'], 'repeat': 1, 'completed': None,
    'tool_calls': None, 'input_tokens': None, 'output_tokens': None, 'latency_ms': None,
    'retrieval_overhead_ms': None, 'retrieved_memory_ids': [], 'retrieval_relevant': None,
    'repeated_error': None, 'transfer_success': None, 'stale_memory_error': None,
    'answer_path': 'experiments/runs/answers/example.txt',
}
validate_result(example)
print('Result schema is ready. Do not call append_result until the live run.')